<a href="https://colab.research.google.com/github/JosephAFerguson/-UserInterface-Proj2/blob/main/DeepLearningJF.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [82]:
from requests import Request, Session
from requests.exceptions import ConnectionError, Timeout, TooManyRedirects
import json
from datetime import datetime, timedelta

In [83]:
class CryptoEndpoint:
    listingsEndpoint = "https://sandbox-api.coinmarketcap.com/v1/cryptocurrency/listings/latest"
    latestQuotes = "https://sandbox-api.coinmarketcap.com/v2/cryptocurrency/quotes/latest"
    historicalQuotes = "https://pro-api.coinmarketcap.com/v2/cryptocurrency/quotes/historical"

    def __init__(self, apikey) -> None:
        self.headers = {
            'Accepts': 'application/json',
            'X-CMC_PRO_API_KEY': apikey,
        }
        self.coinsInfo = {}
        self.coinsIds = []

    def GetCoinIdentifiers(self):
        session = Session()
        session.headers.update(self.headers)

        response = session.get(url=self.listingsEndpoint, params={"limit": 5})
        data = json.loads(response.text)

        for coin in data.get("data", []):
            self.coinsInfo[coin["slug"]] = coin["id"]
            self.coinsIds.append(coin["id"])

        print(f"Loaded {len(self.coinsInfo)} coins.")
        return (self.coinsInfo, self.coinsIds)

    def GetCoinLatestPrices(self, coin_id):

        session = Session()
        session.headers.update(self.headers)
        response = session.get(url=self.latestQuotes, params={"id": coin_id})
        data = json.loads(response.text)

        prices = {}
        for coin_id, info in data.get("data", {}).items():
            quote = info["quote"]["USD"]["price"]
            prices[coin_id] = quote

        return prices

    def GetSampleCoinHistoricalData(self, coin_id, days=4):
        session = Session()
        session.headers.update(self.headers)

        end_time = datetime.utcnow()
        start_time = end_time - timedelta(days=days)

        prices = {}

        params = {
            "id": coin_id,
            "time_start": start_time.isoformat(),
            "time_end": end_time.isoformat(),
            "interval": "24h",
        }

        response = session.get(url=self.historicalQuotes, params=params)
        data = json.loads(response.text)
        print(data["status"]["error_code"])
        coin_data = data.get("data", {})

        historicals = []

        for quote in coin_data["quotes"]:
            date = quote.get("timestamp") or quote.get("time_open")
            price = quote["quote"]["USD"]["price"]
            volume = quote["quote"]["USD"]["volume_24h"]
            historicals.append([price,volume])

        returnData = {coin_id: historicals}
        return returnData

In [84]:
ce = CryptoEndpoint(input("Enter API-KEY"))
(coinsInfo, coinsIds) = ce.GetCoinIdentifiers()
data = []
for coinId in coinsIds:
  coinSample = ce.GetSampleCoinHistoricalData(coinId)
  if len(list(coinSample.values())[0]) < 1:
    continue
  data.append(coinSample)
print(data)

Enter API-KEYa76bd6fc-2b66-4a25-843f-321def3437bd
Loaded 10 coins.
0


/tmp/ipython-input-1105338339.py:46: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  end_time = datetime.utcnow()


0
0
0
0
0
0
0
0
0
[{1321: [[15.04474721138631, 153693757.04], [14.064492575603706, 194103939.46], [14.513608307187168, 115334913.19]]}]
